# Current RCWA API: one immutable problem, repeated runs

Use the public `wavebit.rcwa` module with shared `Material` and explicit `Units`. This notebook runs on NumPy/CPU. Geometry and vacuum wavelengths below are in micrometres, angles are radians, and field output is E in V/m and H in A/m. Layers run bottom to top; z starts at the bottom interface.

The small Fourier order and 64-point NVF grid keep this example quick. They do **not** establish convergence or recommend production resolution. See the [current RCWA guide](../../docs/getting-started/rcwa.md).

In [ ]:
import numpy as np
from wavebit import Material, Units, Job, Execution, rcwa

problem = rcwa.Problem(
    units=Units(length='um'),
    stack=rcwa.Stack(
        period=(.6, .6),
        top=Material(epsilon=1.),
        bottom=Material(epsilon=2.25),
        layers=(rcwa.Layer(
            thickness=.18,
            background=Material(epsilon=1.),
            regions=(rcwa.Region(
                geometry=rcwa.Circle(radius=.12),
                material=Material(epsilon=12.25)),)),)),
    excitation=rcwa.Excitation(
        wavelength=np.array([.85, .9, .95]),
        sources=(rcwa.PlaneWave(side='top', polarization='s', amplitude=1.),)),
    outputs=rcwa.Outputs(fields=True))

## Check, prepare and read the result

The problem owns the structure, illumination and requested output. Numerical approximation and execution settings are separate. `R` and `T` retain `(wavelength, order_v, order_u)` axes. Field components are `(Ex, Ey, Ez, Hx, Hy, Hz)`.

In [ ]:
# These small orders make this a quick demonstration, not a convergence claim.
numerics = rcwa.Numerics(orders=(1, 1), factorization='nvf', shape_nvf_grid_size=64)
report = rcwa.check(problem, numerics=numerics, backend="numpy")
report.raise_if_invalid()
prepared = rcwa.prepare(problem, numerics=numerics, backend="numpy")
result = prepared.run(problem)
print(result.R.shape)  # (3, 3, 3): wavelength, order_v, order_u
print(result.transmission.order(0, 0).efficiency)
print(result.transmission.at_wavelength_index(1).wavelengths)
samples = result.field(x=np.array([0., .1]), y=np.array([0.]), z=np.array([.09]))
print(samples.values.shape)  # (3, 1, 1, 2, 6)
print(samples.units)         # E: V/m, H: A/m, coordinates: um

## Reuse with the complete live problem

Replace only compatible live inputs and pass the complete new problem to `prepared.run`. Earlier results retain their own fields; no mutable last-run state chooses the next input. Grid/mode-layout changes require new preparation. `Job` records the problem, numerics and execution settings as portable JSON.

In [ ]:
new_problem = problem.replace(excitation=problem.excitation.replace(
    wavelength=np.array([.86, .91, .96])))
new_result = prepared.run(new_problem)
# result and its retained fields still belong to the first calculation.
job = Job(solver='rcwa', problem=problem, numerics=numerics,
          execution=Execution(backend='numpy'))
assert Job.from_json(job.to_json()).key == job.key
print(new_result.T.sum(axis=(1, 2)))

Continue with the [shared lifecycle](../../docs/guide/common/lifecycle.md), [materials](../../docs/guide/common/materials.md) and [RCWA curriculum](../../docs/learn/fundamentals/index.md). Numbered advanced notebooks in this folder explicitly retain the compatibility interface.